# MLP-045 — embeddings категориальных признаков

Этот эксперимент сравнивает принятую **MLP-024** с той же скрытой сетью, но
заменяет one-hot представление всех шести категориальных признаков на обучаемые
`nn.Embedding`.

Путь данных внутри каждого fold:

`train категории → OrdinalEncoder → индексы → nn.Embedding → объединение с числами → MLP`.

Словари категорий fit-ятся только на train-части fold. Индекс `0` зарезервирован
для категории, которая не встречалась при fit. `test.csv` не участвует в CV.


## 1. Контролируемое изменение

Источник истины: `src/ml_project/mlp_experiments/mlp_045_embeddings.py`.

В эксперименте зафиксированы признаки и скрытая часть MLP-024. Изменяется способ
представления категорий:

- `Embarked`: embedding dimension 2;
- `FamilySizeGroup`: embedding dimension 3;
- `IsnotAlone`: embedding dimension 1;
- `CabinKnown`: embedding dimension 1;
- `Deck`: embedding dimension 3;
- `SexPclass`: embedding dimension 3.

Даже бинарные `IsnotAlone` и `CabinKnown` проходят через embedding, чтобы
единственным изменением относительно MLP-024 была замена one-hot механизма.
Не создавайте словари категорий вручную по всей таблице: их строит fold-fitted
`OrdinalEncoder`.


In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import ConfusionMatrixDisplay

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    p for p in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (p / "README.md").is_file() and (p / "src").is_dir()
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_project import DataCatalog
import ml_project.config as project_config
import ml_project.mlp_experiment as mlp_tools

EXPERIMENT_MODULE = 'ml_project.mlp_experiments.mlp_045_embeddings'
mlp_tools = importlib.reload(mlp_tools)
experiment_module = mlp_tools.load_mlp_experiment(EXPERIMENT_MODULE)
spec = experiment_module.EXPERIMENT

print("Project:", PROJECT_ROOT)
print("Experiment:", spec.experiment_id, spec.title)
print("Module:", EXPERIMENT_MODULE)
print("Hypothesis:", spec.hypothesis)
print("Controlled change:", spec.change_description)
print("Reference:", spec.parent_mlp_module or spec.sklearn_reference_module or "logistic baseline")
display(pd.Series(vars(spec.training), name="value").to_frame())


## 2. Данные и роли признаков

Числовые признаки заполняются и масштабируются внутри fold. Категориальные
признаки там же получают целочисленные индексы. Неизвестная validation/test
категория кодируется как `-1`, после чего модель сдвигает её в специальный
embedding-индекс `0`.


In [ ]:
catalog = DataCatalog(PROJECT_ROOT, project_config.RAW_DIR, project_config.DATASETS)
catalog.validate()
raw_train = catalog.load(project_config.TRAIN_DATASET)
raw_test = catalog.load(project_config.INFERENCE_DATASET)
train_snapshot = raw_train.copy(deep=True)
test_snapshot = raw_test.copy(deep=True)

prepared = experiment_module.prepare_features(raw_train.copy(deep=True))
prepared_test = experiment_module.prepare_features(raw_test.copy(deep=True))
pd.testing.assert_frame_equal(raw_train, train_snapshot)
pd.testing.assert_frame_equal(raw_test, test_snapshot)
mlp_tools.validate_feature_data(
    prepared, raw_train, target=project_config.TARGET, key=project_config.KEY,
)

if project_config.TARGET in prepared_test.frame:
    raise ValueError("Inference data must not acquire target")
if not prepared_test.frame.index.equals(raw_test.index):
    raise ValueError("prepare_features changed inference row index/order")
if prepared.numeric_features != prepared_test.numeric_features:
    raise ValueError("Train/test numeric feature declarations differ")
if prepared.categorical_features != prepared_test.categorical_features:
    raise ValueError("Train/test categorical feature declarations differ")
missing_test = [
    name for name in prepared.features
    if name not in prepared_test.frame and name not in prepared.fold_generated_features
]
if missing_test:
    raise KeyError(f"Inference is missing features: {missing_test}")

feature_report = pd.DataFrame({
    "role": ["numeric"] * len(prepared.numeric_features)
            + ["categorical"] * len(prepared.categorical_features),
    "feature": list(prepared.features),
}).set_index("feature")
train_feature_view = prepared.frame.reindex(columns=list(prepared.features))
test_feature_view = prepared_test.frame.reindex(columns=list(prepared.features))
feature_report["dtype"] = train_feature_view.dtypes.astype(str)
feature_report["missing_train"] = train_feature_view.isna().sum()
feature_report["missing_test"] = test_feature_view.isna().sum()
display(feature_report)
display(
    prepared.frame.reindex(
        columns=[project_config.KEY, *prepared.features, project_config.TARGET]
    ).head()
)


## 3. Preview индексов и embedding-слоёв

Эта диагностическая ячейка показывает изученные категории, их индексы, размеры
embedding-таблиц и форму входа. Официальный CV ниже создаёт всё заново отдельно
для каждого train fold.


In [ ]:
preview_transformer_builder = getattr(
    experiment_module, "build_fold_transformer", None
)
preview_transformer = (
    preview_transformer_builder() if preview_transformer_builder else None
)
preview_frame = prepared.frame.copy(deep=True)
if preview_transformer is not None:
    preview_transformer.fit(preview_frame)
    preview_frame = preview_transformer.transform(preview_frame)

preview_preprocessor = mlp_tools.build_preprocessor(prepared, spec.training)
preview_x = np.asarray(
    preview_preprocessor.fit_transform(
        preview_frame.loc[:, list(prepared.features)]
    ),
    dtype=np.float32,
)
input_dim = preview_x.shape[1]
preview_model = experiment_module.build_network(input_dim)

ordinal_encoder = (
    preview_preprocessor.named_transformers_["categorical"]
    .named_steps["encoder"]
)
mapping_rows = []
for feature, categories in zip(
    prepared.categorical_features,
    ordinal_encoder.categories_,
    strict=True,
):
    mapping_rows.append({
        "feature": feature,
        "unknown_index": 0,
        "known_categories": list(categories),
        "model_indices": list(range(1, len(categories) + 1)),
    })
display(pd.DataFrame(mapping_rows))

parameter_count = sum(
    parameter.numel()
    for parameter in preview_model.parameters()
    if parameter.requires_grad
)
print("Numeric columns:", prepared.numeric_features)
print("Embedded columns:", prepared.categorical_features)
print("Preprocessed columns:", input_dim)
print("Combined numeric + embedding width:", 16)
print("Trainable parameters:", parameter_count)
print(preview_model)

with torch.no_grad():
    sample_logits = preview_model(
        torch.tensor(preview_x[:5], dtype=torch.float32)
    )
print("Five preview logits:", sample_logits.tolist())


## 4. Outer CV

Каждый outer fold создаёт новый preprocessor и новую сеть. Внутри outer-train
выделяется inner validation только для early stopping. OOF-предсказание каждой
строки сделано моделью, которая эту строку не видела.


In [ ]:
result = mlp_tools.run_mlp_cv(
    prepared,
    target=project_config.TARGET,
    key=project_config.KEY,
    build_network=experiment_module.build_network,
    spec=spec,
    build_fold_transformer=getattr(experiment_module, "build_fold_transformer", None),
)
display(result.fold_metrics.round(4))
display(result.summary.round(4))
print("Encoded input dimensions by fold:", result.input_dims)


In [ ]:
figure = mlp_tools.plot_training_histories(result)
display(figure)
plt.close(figure)


## 5. Reference champion и paired-сравнение

Reference заново обучается на тех же outer folds. Для первого MLP это последний
принятый sklearn-чемпион; для следующих опытов — последний MLP с
`decision: adopt`. Положительный `improvement` всегда означает улучшение,
включая `log_loss`, где меньше — лучше.


In [ ]:
if spec.parent_mlp_module:
    reference = mlp_tools.run_mlp_parent_reference(
        raw_train,
        module_name=spec.parent_mlp_module,
        target=project_config.TARGET,
        key=project_config.KEY,
        cv_splits=result.cv_splits,
    )
else:
    reference = mlp_tools.run_sklearn_champion_reference(
        PROJECT_ROOT,
        raw_train,
        module_name=spec.sklearn_reference_module,
        target=project_config.TARGET,
        key=project_config.KEY,
        feature_groups=project_config.FEATURE_GROUPS,
        cv_splits=result.cv_splits,
    )

comparison = mlp_tools.compare_mlp_with_reference(result, reference, spec)
display(comparison.summary.round(4))
display(comparison.criteria.round(4))
display(
    comparison.paired_deltas[
        comparison.paired_deltas["metric"].eq(spec.primary_metric)
    ].round(4)
)


In [ ]:
oof = result.oof_predictions
ConfusionMatrixDisplay.from_predictions(
    oof["target"], oof["prediction"], labels=[0, 1],
    display_labels=["Not survived", "Survived"], cmap="Blues", colorbar=False,
)
plt.title(f"{spec.experiment_id}: OOF confusion matrix")
plt.tight_layout()
plt.show()

analysis = raw_train[[project_config.KEY, "Sex", "Pclass", "Name"]].merge(
    oof, on=project_config.KEY, validate="one_to_one"
)
segment_metrics = analysis.groupby("Sex", observed=True).apply(
    lambda part: pd.Series({
        "rows": len(part),
        "accuracy": (part["target"] == part["prediction"]).mean(),
        "mean_probability": part["probability"].mean(),
    }),
    include_groups=False,
)
display(segment_metrics)
display(
    analysis.assign(confidence=lambda x: np.maximum(x.probability, 1-x.probability))
    .query("target != prediction")
    .sort_values("confidence", ascending=False)
    .head(15)
)

candidate_oof = result.oof_predictions.set_index(project_config.KEY)
reference_oof = reference.oof_predictions.set_index(project_config.KEY)
error_changes = candidate_oof[["target", "prediction", "probability"]].join(
    reference_oof[["prediction", "probability"]].rename(columns={
        "prediction": "reference_prediction",
        "probability": "reference_probability",
    }),
    validate="one_to_one",
)
print("Исправлено ошибок reference:", int((
    (error_changes.prediction == error_changes.target)
    & (error_changes.reference_prediction != error_changes.target)
).sum()))
print("Добавлено новых ошибок:", int((
    (error_changes.prediction != error_changes.target)
    & (error_changes.reference_prediction == error_changes.target)
).sum()))


## 6. Вывод до сохранения

Запишите:

- победила ли MLP-045 по accuracy и guardrails;
- сколько folds она выиграла у MLP-024;
- исправленные и добавленные OOF-ошибки;
- поведение train/inner-validation loss;
- оправдалось ли обучение плотных представлений на категориях размером 3–8.

Не оценивайте embeddings по Kaggle test: у него нет target.


## 7. Сохранить результат и Obsidian-карточку

Сохраняются fold metrics, OOF, paired Δ, история эпох и hashes. Затем создаются
карточка `mlp-experiments/MLP-xxx ...md`, CSV-реестр, индекс и блок в README.
Ручной анализ карточки и поле `decision` при перезапуске сохраняются.


In [ ]:
run_dir = None
if spec.save_artifacts:
    dataset_path = catalog.path(project_config.TRAIN_DATASET)
    run_dir = mlp_tools.save_mlp_run(
        PROJECT_ROOT, experiment_module, prepared, result,
        target=project_config.TARGET, key=project_config.KEY,
        dataset_path=dataset_path,
        comparison=comparison,
    )
    print("Artifacts:", run_dir.relative_to(PROJECT_ROOT))
    if spec.sync_docs:
        note_path = mlp_tools.sync_mlp_report(
            PROJECT_ROOT,
            experiment_module,
            prepared,
            result,
            comparison,
            run_dir,
            dataset_path=dataset_path,
        )
        print("Obsidian card:", note_path.relative_to(PROJECT_ROOT))
else:
    print("Artifact saving is disabled in EXPERIMENT.")


## 8. Финальная модель и Kaggle preview

Финальная сеть обучается на всей размеченной таблице; внутри неё остаётся inner
split для выбора эпохи. Это отдельный fit после оценки CV. `SAVE_FINAL_MODEL`
управляет записью весов, preprocessor и CSV. Сначала изучите CV и только затем
включайте сохранение.


In [ ]:
SAVE_FINAL_MODEL = False

if SAVE_FINAL_MODEL:
    if run_dir is None:
        raise RuntimeError("Enable spec.save_artifacts before final model saving")
    final_model = mlp_tools.fit_final_mlp(
        prepared,
        target=project_config.TARGET,
        build_network=experiment_module.build_network,
        spec=spec,
        build_fold_transformer=getattr(experiment_module, "build_fold_transformer", None),
    )
    test_probability = mlp_tools.predict_prepared_mlp(final_model, prepared_test)
    submission = pd.DataFrame({
        project_config.KEY: prepared_test.frame[project_config.KEY].to_numpy(),
        project_config.TARGET: (
            test_probability >= spec.training.threshold
        ).astype("int64"),
    })
    if len(submission) != len(raw_test) or not submission[project_config.KEY].equals(raw_test[project_config.KEY]):
        raise ValueError("Submission row/key contract failed")
    mlp_tools.save_final_mlp(run_dir, final_model)
    submission.to_csv(run_dir / "submission.csv", index=False)
    pd.DataFrame({
        project_config.KEY: prepared_test.frame[project_config.KEY],
        "probability": test_probability,
    }).to_csv(run_dir / "test_probabilities.csv", index=False)
    display(submission.head())
    print("Final model and submission saved to:", run_dir.relative_to(PROJECT_ROOT))
else:
    print("Final fit is disabled. Set SAVE_FINAL_MODEL = True after accepting CV result.")


## 9. Решение и следующий эксперимент

После анализа измените в карточке только `decision:` на `adopt`, `reject`,
`iterate` или `inconclusive`, затем выполните:

```powershell
.\sync-MLPexperiment-state.cmd
```

Из корня проекта:

```powershell
.\new-MLPexperiment.cmd
```

Команда создаст следующий `MLP-xxx`, отдельный модуль и notebook. Старый модуль
после измеренного запуска не редактируйте: он фиксирует точное определение опыта.
